# Gradio prototype for Pandas crystal ball

This notebook contains a basic Gradio UI prototype for the pandas crystal ball project from the primer.<br>
It allows for selecting between a nano and a mini gpt-4.1 model.

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr

In [2]:
# constants; I will use them for the model selector between the nano and mini models
# it's only relevant here to showcase the use of selector in Gradio
MODEL_GPT = 'gpt-4.1-nano-2025-04-14'
MODEL_GPT2 = 'gpt-4.1-mini'

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

API key looks good so far


In [4]:
# the system message that allows us to make the tool useful solely for pandas knowledge
system_message = """
You are a data scientist assistant. You answer questions regarding a Python library called pandas.
If you are asked about anything that is not related to that library, refuse to answer the question.
"""


In [5]:
def talker(message: str) -> bytes:
    """
    The goal of this function is to transform the LLM response text to voice.
    Args:
        message: the response of the LLM
    Returns:
        response.content: raw bytes representing a voice track
    """
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    
      input=message
    )
    return response.content


def get_answer_stream(question: str, txtmodel_version: str) -> [str, bytes]:
    """
    The goal of this function is to get the LLM response for a given question.
    Args:
        question: contains the question asked to the LLM
        txtmodel_version: defines which model to use
    Returns:
        reply: the output generated by the LLM
        voice: raw bytes representing a voice track of the reply
    """
    if txtmodel_version == 'nano':
        stream = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": question}
                ],
            )
    else:
        stream = openai.chat.completions.create(
            model=MODEL_GPT2,
            messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": question}
                ],
            )

    reply = f'using {txtmodel_version} version of chatgpt' + '\n' + stream.choices[0].message.content
    voice = talker(reply)

    return reply, voice

In [7]:
# the following block of code shows a very simple UI generated with Gradio
# it contains two textboxes, one to input the message, the other for the output
# it also contains a selector for the model, a button to submit the question
# and a basic audio player for our voice track

with gr.Blocks() as ui:
    with gr.Row():
        message_input = gr.Textbox(label="Your message:", info="Enter a pandas-related question", lines=5)
        model_selector = gr.Dropdown(["nano", "mini"], label="Select model", value="nano")
    with gr.Row():
        submission_button = gr.Button("Submit")
    with gr.Row():
        message_output = gr.Textbox(label="Ask the pandas crystal ball a question:", lines=5)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    
    submission_button.click(get_answer_stream, inputs=[message_input, model_selector], outputs=[message_output, audio_output])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
